In [1]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
import random
from collections import Counter

# Custom PyTorch Dataset class to tokenize and handle data
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize the input text
        encodings = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(0),
            "attention_mask": encodings["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float)
        }
    
# Load the IMDB dataset from HuggingFace
dataset = load_dataset("imdb")

# Load tokenizer for RoBERTa
from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [2]:
# Separate texts and labels
train_data = dataset["train"]
test_data = dataset["test"]

# Init number of sample
nb_train_sample = 100
nb_test_sample = 100

# Filter by label
# Subsample the dataset (first 2000 for train, 1000 for test)
train_pos = [ex for ex in train_data if ex["label"] == 1][:int(nb_train_sample/2)] #list of dictionary, each dict contains text and label
train_neg = [ex for ex in train_data if ex["label"] == 0][:int(nb_train_sample/2)]
test_pos = [ex for ex in test_data if ex["label"] == 1][:int(nb_test_sample/2)]
test_neg = [ex for ex in test_data if ex["label"] == 0][:int(nb_test_sample/2)]

# Combine and shuffle
train_subset = train_pos + train_neg #list of dictionary, each dict contains text and label
test_subset = test_pos + test_neg

random.shuffle(train_subset)
random.shuffle(test_subset)

# Extract texts and labels
train_texts = [ex["text"] for ex in train_subset]
train_labels = [ex["label"] for ex in train_subset]
test_texts = [ex["text"] for ex in test_subset]
test_labels = [ex["label"] for ex in test_subset]

In [3]:
from sklearn.model_selection import train_test_split
# Stratified split: 80% train, 20% val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, stratify=train_labels, random_state=42
)

# Convert dataset into PyTorch DataLoaders
train_dataset = IMDBDataset(train_texts, train_labels, tokenizer)
val_dataset = IMDBDataset(val_texts, val_labels, tokenizer)
test_dataset = IMDBDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [4]:
print(Counter(train_dataset.labels))
print(Counter(val_dataset.labels))
print(Counter(test_dataset.labels))

Counter({0: 40, 1: 40})
Counter({1: 10, 0: 10})
Counter({1: 50, 0: 50})


In [5]:
for batch in train_loader:
    break
print({k: v.shape for k,v in batch.items()})

{'input_ids': torch.Size([32, 128]), 'attention_mask': torch.Size([32, 128]), 'label': torch.Size([32])}


In [15]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# Define the compute_metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = torch.argmax(pred.predictions, axis=1)  # Get the predicted class index
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    logging_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
)

# Load the dataset and tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define the model
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model.config.problem_type = "single_label_classification"

# Create the Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
10,0.690600
20,0.708600
30,0.685000
40,0.642100
50,0.603100


TrainOutput(global_step=50, training_loss=0.665888843536377, metrics={'train_runtime': 488.1255, 'train_samples_per_second': 0.819, 'train_steps_per_second': 0.102, 'total_flos': 26311105536000.0, 'train_loss': 0.665888843536377, 'epoch': 5.0})

In [17]:
trainer.evaluate(test_dataset)

/var/folders/hs/jhbbsd052zz044qtbzkkhxl80000gn/T/ipykernel_36745/813325653.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "label": torch.tensor(self.labels[idx], dtype=torch.float)


{'eval_loss': 0.6469647288322449,
 'eval_accuracy': 0.86,
 'eval_runtime': 7.2601,
 'eval_samples_per_second': 13.774,
 'eval_steps_per_second': 1.791,
 'epoch': 5.0}

In [18]:
model.save_pretrained("roberta_finetuned")
tokenizer.save_pretrained("roberta_finetuned")

('roberta_finetuned/tokenizer_config.json',
 'roberta_finetuned/special_tokens_map.json',
 'roberta_finetuned/vocab.json',
 'roberta_finetuned/merges.txt',
 'roberta_finetuned/added_tokens.json')

In [ ]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer

# Path to your directory
model_path = "roberta_finetuned"

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained(model_path)

# Load model (with safetensors)
model = RobertaForSequenceClassification.from_pretrained(model_path, use_safetensors=True)

# Example usage
text = "This is amazing, but I don't like it!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)

# Get prediction
pred = outputs.logits.argmax(dim=1)
print(f"Predicted class: {pred.item()}")

Predicted class: 0
